In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
dataset = pd.read_csv("IMDB Dataset.csv")

In [3]:
dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df = dataset.copy()

In [5]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [6]:
df['sentiment'].value_counts()

positive    25000
negative    25000
Name: sentiment, dtype: int64

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
df.duplicated().sum()

418

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.duplicated().sum()

0

In [11]:
# Basic Preprocessing
# Remove HTML tags
# Lowercase
# Remove Stopwords

In [12]:
import re
def remove_tags(raw_text):
    return re.sub(re.compile('<.*?>'), '', raw_text)

In [13]:
df['review'] = df['review'].apply(remove_tags)

In [14]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [15]:
df['review'] = df['review'].apply(lambda x: x.lower())

In [18]:
import nltk
from nltk.corpus import stopwords
# nltk.download('stopwords')

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list])\
    .apply(lambda x: " ".join(x))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\GLOBAL-3\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [19]:
X = df['review']
y = df['sentiment']

In [20]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
X_train.shape

(39665,)

In [23]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000)

In [24]:
X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

In [25]:
X_train_bow.shape

(39665, 5000)

In [26]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)

GaussianNB()

In [27]:
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score, classification_report
accuracy_score(y_test, y_pred)

0.7531511545830393

In [28]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.86      0.78      4939
           1       0.83      0.64      0.72      4978

    accuracy                           0.75      9917
   macro avg       0.77      0.75      0.75      9917
weighted avg       0.77      0.75      0.75      9917



In [29]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train_bow, y_train)

y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.8384592114550772

In [30]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, ngram_range=(1,2))

X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

rf = RandomForestClassifier()
rf.fit(X_train_bow, y_train)

y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.8432993848946254

### TFIDF

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [32]:
tfidf = TfidfVectorizer()

X_train_tfidf = cv.fit_transform(X_train).toarray()
X_test_tfidf = cv.transform(X_test).toarray()

rf = RandomForestClassifier()
rf.fit(X_train_tfidf, y_train)

y_pred = rf.predict(X_test_tfidf)
accuracy_score(y_test, y_pred)

0.8452152868811132

### Word2Vec

In [35]:
import gensim

In [36]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [39]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [40]:
model = gensim.models.Word2Vec(
        window=10,
     min_count=2
)

In [41]:
model.build_vocab(story)

In [42]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(29541279, 30891345)

In [43]:
len(model.wv.index_to_key)

61843

In [44]:
def document_vector(doc):
    # remove oov words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [45]:
document_vector(df['review'].values[5])

array([ 7.47563466e-02,  2.88118690e-01, -1.09560706e-01, -1.51439428e-01,
        1.14385933e-01, -5.08916140e-01,  1.37292549e-01,  4.25886929e-01,
        1.32111490e-01,  1.87478319e-01,  3.52128483e-02, -3.25990111e-01,
        4.35931712e-01,  4.30061579e-01,  3.23717654e-01,  1.46225691e-01,
       -6.13496453e-02,  4.15534407e-01, -2.87307687e-02, -5.78260899e-01,
       -3.32269371e-02, -4.22200412e-01, -1.45939946e-01, -3.13905883e-03,
       -4.19035167e-01, -2.74525881e-01, -2.06027329e-01,  2.31909931e-01,
       -7.52498507e-01, -4.28522944e-01,  2.52282917e-01,  1.84620589e-01,
       -3.01350560e-02, -2.74036020e-01,  4.39981371e-01,  3.80380079e-02,
        1.85023233e-01, -4.67277288e-01, -2.64157772e-01, -3.83308972e-03,
       -3.41426969e-01, -8.41191411e-02, -1.90378994e-01, -4.67294872e-01,
        2.16106102e-01,  2.70571783e-02, -2.83642083e-01,  4.31802303e-01,
       -4.26383913e-01,  5.70439696e-01,  1.27935305e-01,  2.63517767e-01,
       -2.53666490e-01,  

In [46]:
from tqdm import tqdm

In [47]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

 62%|██████▏   | 30908/49582 [1:25:42<51:46,  6.01it/s]  


ValueError: need at least one array to concatenate

In [49]:
X = np.array(X)

In [50]:
X.shape

(30908, 100)

In [55]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment'])[:30908]

In [56]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [57]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)

0.8188288579747655

In [ ]:
# Pretrained Word2Vec